# Student Dropout Intelligence System

**Goal (2 hours):** Use the **same dataset** to answer **three different questions**:

1) **K-Means** → *What types of students exist?* (no labels)  
2) **Logistic Regression** → *Who might drop out?* (explainable probability)  
3) **Random Forest** → *Who might drop out in messy reality?* (stronger patterns)

> Remember: **Models don’t compete. Questions compete.**


## 1) Import libraries (why?)

- **pandas/numpy** → data handling
- **scikit-learn** → ML models
- **matplotlib/seaborn** → simple visuals

If a library is missing, install it using:
`pip install pandas numpy scikit-learn matplotlib seaborn`


In [21]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


ModuleNotFoundError: No module named 'pandas'

## 2) Load the dataset (why?)

- We load **student_data.csv** from the same folder.
- Each row is **one student**.
- Each column is a **behavior signal** from the first week.


In [ ]:
df = pd.read_csv("student_data.csv")
df.head()


## 3) Quick scan (why?)

We check:
- column names and types
- whether data looks reasonable
- basic statistics (min/max/average)

This is how engineers avoid surprises later.


In [ ]:
df.info()


In [ ]:
df.describe()


## 4) Define the feature set (why?)

We choose only the **input signals** we want the models to learn from.
We keep the `dropout` column separate because it's the *outcome label* (used only for supervised learning).


In [ ]:
features = [
    "attendance_rate",
    "assignment_submitted",
    "avg_quiz_score",
    "coding_minutes",
    "doubts_asked",
    "late_submissions",
    "project_commit_count",
    "peer_messages_sent",
    "video_watched_percent",
    "background"
]


## 5) Standardize features (why?)

K-Means uses **distance**.  
If one column has large values (like `coding_minutes`) it can dominate the result.

So we scale all features to the same level (mean 0, std 1).


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])


# PART A — K-Means (Discovery)

## 6) Run K-Means (why?)

K-Means answers:
> **“What types of students exist?”**

No dropout labels needed.  
It groups students by similarity of behavior.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)

df["cluster"].value_counts()


## 7) Understand each cluster (why?)

This is the **WOW moment**.

We take the average behavior inside each cluster.
Then we *name* the clusters like personas:
- Silent grinders
- Social learners
- At-risk drifters
- Consistent performers


In [ ]:
cluster_profile = df.groupby("cluster")[features].mean()
cluster_profile


### Optional: quick visual (why?)

This makes it easier to “see” differences between clusters.


In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(data=df, x="cluster")
plt.title("How many students are in each cluster?")
plt.show()


# PART B — Logistic Regression (Explainable Risk)

## 8) Train/test split (why?)

We split the data:
- **train** → model learns patterns
- **test** → model is checked on unseen rows

This prevents “memorizing”.


In [ ]:
X = df[features]
y = df["dropout"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 9) Train Logistic Regression (why?)

Logistic Regression answers:
> **“Will student X drop out?”** (with a probability)

It’s great when we want:
- fast baseline
- explainable reasoning (weights)


In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

df["dropout_risk_lr"] = lr.predict_proba(X)[:, 1]
df[["student_id", "dropout_risk_lr"]].head(10)


## 10) Explain 'why' using coefficients (why?)

Coefficients show which signals push risk up or down.

- positive → increases dropout risk  
- negative → reduces dropout risk


In [ ]:
lr_weights = pd.Series(lr.coef_[0], index=features).sort_values()
lr_weights


In [ ]:
plt.figure(figsize=(8,5))
lr_weights.plot(kind="barh")
plt.title("Logistic Regression – Risk Factors (Explainable)")
plt.show()


# PART C — Random Forest (Messy Reality)

## 11) Train Random Forest (why?)

Random Forest answers the same question as Logistic Regression:
> “Will student X drop out?”

But it is better when:
- patterns are non-linear
- human behavior is inconsistent
- you want stronger performance


In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

df["dropout_risk_rf"] = rf.predict_proba(X)[:, 1]
df[["student_id", "dropout_risk_rf"]].head(10)


## 12) Feature importance (why?)

Random Forest doesn’t give simple weights like Logistic Regression,
but it can still show which features mattered most overall.


In [ ]:
rf_imp = pd.Series(rf.feature_importances_, index=features).sort_values()
rf_imp


In [ ]:
plt.figure(figsize=(8,5))
rf_imp.plot(kind="barh")
plt.title("Random Forest – Feature Importance")
plt.show()


# FINAL — Put it together (System View)

## 13) One student, three views (why?)

- `cluster` → what type of student?
- `dropout_risk_lr` → explainable risk score
- `dropout_risk_rf` → stronger risk score for messy patterns

This is what “intelligence system” looks like.


In [ ]:
df[[
    "student_id",
    "cluster",
    "dropout_risk_lr",
    "dropout_risk_rf",
    "dropout"
]].sort_values("dropout_risk_rf", ascending=False).head(12)


## 14) Simple action rules (optional)

Now we simulate a *real* intervention policy:

- High risk + at-risk cluster → call immediately
- Medium risk + social cluster → accountability buddy
- High risk + silent grinders → check-in only (they might be fine)


In [ ]:
def recommend_action(cluster, risk):
    # You can rename clusters later after you inspect cluster_profile
    if risk >= 0.75 and cluster in [2, 3]:
        return "CALL (high risk)"
    if risk >= 0.60 and cluster == 1:
        return "ACCOUNTABILITY BUDDY"
    if risk >= 0.75 and cluster == 0:
        return "CHECK-IN (silent grinder)"
    if risk >= 0.60:
        return "FOLLOW-UP MESSAGE"
    return "NO ACTION"

df["action"] = df.apply(lambda r: recommend_action(r["cluster"], r["dropout_risk_rf"]), axis=1)
df[["student_id", "cluster", "dropout_risk_rf", "action"]].sort_values("dropout_risk_rf", ascending=False).head(15)


### Final takeaway

> **Machine learning is not picking the best algorithm.  
> Machine learning is asking the right question.**
